In [7]:
import pandas as pd
df = pd.read_csv(r"C:\Users\srina\Coding\Chess-Bot\lichess-bot\engines\train_eval\game-data\training_processed.csv")
df.head()

,game_id,date,ply,move_uci,current_fen,next_fen,is_chosen_move,piece_type,is_capture,is_check,...,delta_position_enemy_king_zone_rook_attackers,delta_position_enemy_king_zone_queen_attackers,delta_position_king_to_enemy_king_distance,delta_position_king_to_friendly_pawn_distance,delta_position_king_to_enemy_pawn_distance,delta_position_king_tropism,delta_position_controlled_square_count,delta_position_space_in_enemy_half,delta_position_center_control_count,delta_position_enemy_controlled_square_count
0,1,2026-04-04 00:00:00,0,e2e4,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,1,p,0,0,...,0,0,0,0,-2,0.083333,0,0,0,7
1,1,2026-04-04 00:00:00,0,g2g3,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/8/6P1/PPPPPP1P/RNBQKBNR ...,0,p,0,0,...,0,0,0,0,-1,0.033333,0,0,0,2
2,1,2026-04-04 00:00:00,0,c2c4,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR ...,0,p,0,0,...,0,0,0,0,-2,0.083333,0,0,0,3
3,1,2026-04-04 00:00:00,0,g1h3,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/8/7N/PPPPPPPP/RNBQKB1R b...,0,n,0,0,...,0,0,0,0,0,0.057143,0,0,0,2
4,1,2026-04-04 00:00:00,0,d2d3,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/8/3P4/PPP1PPPP/RNBQKBNR ...,0,p,0,0,...,0,0,0,0,-1,0.033333,0,0,0,5


In [8]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RANDOM_SEED = 42

In [18]:
print(df.describe())

       is_chosen_move    is_capture      is_check   is_castling  \
count    3.893704e+06  3.893704e+06  3.893704e+06  3.893704e+06   
mean     9.510584e-02  6.231650e-02  3.273053e-02  4.198573e-03   
std      2.933611e-01  2.417295e-01  1.779305e-01  6.466023e-02   
min      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
25%      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
50%      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
75%      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
max      1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00   

       delivers_checkmate      distance  signed_distance   from_square  \
count        3.893704e+06  3.893704e+06     3.893704e+06  3.893704e+06   
mean         6.320460e-04  2.019405e+00     1.379166e+00  3.167396e+01   
std          2.513258e-02  1.248086e+00     1.932256e+00  2.046405e+01   
min          0.000000e+00  1.000000e+00    -9.899495e+00  0.000000e+00   
25%          0.000000e+00 

## ML Pipeline

Prepare the processed dataset, create the train/validation/test split, and train the first XGBoost baseline.

In [9]:
# Drop columns that should not go into the model.
base_drop_columns = [
    "game_id",
    "date",
    "ply",
    "move_uci",
    "current_fen",
    "next_fen",
]
next_position_columns = [col for col in df.columns if col.startswith("next_position_")]

feature_df = df
feature_df.drop(columns=base_drop_columns + next_position_columns, errors="ignore", inplace=True)

# One-hot encode the moving piece type.
piece_type_map = {
    "p": "pawn",
    "n": "knight",
    "b": "bishop",
    "r": "rook",
    "q": "queen",
    "k": "king",
}
raw_piece_type = feature_df.pop("piece_type").astype(str).str.lower().str.strip()
piece_labels = raw_piece_type.map(piece_type_map).fillna(raw_piece_type)
piece_dummies = pd.get_dummies(piece_labels, dtype=np.int8)
expected_piece_columns = ["pawn", "knight", "bishop", "rook", "queen", "king"]
piece_dummies = piece_dummies.reindex(columns=expected_piece_columns, fill_value=0)
piece_dummies.columns = [f"is_{piece}_move" for piece in piece_dummies.columns]
feature_df = pd.concat([feature_df, piece_dummies], axis=1)

# Shuffle rows deterministically before splitting.
feature_df = feature_df.sample(frac=1, random_state=RANDOM_SEED)

# Keep y as a Series through the split so row indices stay aligned with X.
y = feature_df.pop("is_chosen_move").astype(np.int8)
X = feature_df.copy()

print(f"Rows ready for splitting: {len(X):,}")
print(f"Feature columns after preprocessing: {X.shape[1]:,}")

Rows ready for splitting: 3,893,704
Feature columns after preprocessing: 101


In [5]:
def evaluate_split(name: str, X_split: pd.DataFrame, y_split: pd.Series) -> None:
    probabilities = model.predict_proba(X_split)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    accuracy = accuracy_score(y_split, predictions)
    balanced_accuracy = balanced_accuracy_score(y_split, predictions)
    roc_auc = roc_auc_score(y_split, probabilities) if len(np.unique(y_split)) > 1 else float("nan")
    confusion = confusion_matrix(y_split, predictions)

    print(f"\n{name} results")
    print(f"  Accuracy: {accuracy * 100:.2f}%")
    print(f"  Balanced accuracy: {balanced_accuracy * 100:.2f}%")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  Positive rate: {y_split.mean() * 100:.2f}%")
    print("  Confusion matrix:")
    print(confusion)
    print("  Classification report:")
    print(classification_report(y_split, predictions, digits=4, zero_division=0))

# Note: evaluation calls are executed after the model is trained to ensure variables exist.


In [10]:
# 70% / 15% / 15% split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=y_temp,
)

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert X_test.index.equals(y_test.index)

# Standardize numeric columns only, excluding one-hot and binary label-like columns.
non_scaled_columns = {
    "is_capture",
    "is_check",
    "is_castling",
    "delivers_checkmate",
    "avoids_capture",
    "is_pawn_move",
    "is_knight_move",
    "is_bishop_move",
    "is_rook_move",
    "is_queen_move",
    "is_king_move",
}
numeric_columns = [
    column
    for column in X_train.columns
    if column not in non_scaled_columns and pd.api.types.is_numeric_dtype(X_train[column])
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_val_scaled[numeric_columns] = scaler.transform(X_val[numeric_columns])
X_test_scaled[numeric_columns] = scaler.transform(X_test[numeric_columns])

print(f"Train/Val/Test sizes: {len(X_train_scaled):,} / {len(X_val_scaled):,} / {len(X_test_scaled):,}")
print(f"Scaled numeric columns: {len(numeric_columns):,}")

Train/Val/Test sizes: 2,725,592 / 584,056 / 584,056
Scaled numeric columns: 90


In [13]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    tree_method="hist",
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight = 10
)

model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False,
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [15]:
from pathlib import Path
import json
import joblib

artifact_dir = Path(r"C:\Users\srina\Coding\Chess-Bot\lichess-bot\engines\train_eval\model_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "policy_xgb.joblib"
metadata_path = artifact_dir / "policy_xgb_metadata.json"

bundle = {
    "model": model,
    "scaler": scaler,
    "feature_columns": list(X_train_scaled.columns),
    "numeric_columns": list(numeric_columns),
    "random_seed": RANDOM_SEED,
    "threshold": 0.5,
}

joblib.dump(bundle, artifact_path)

metadata = {
    "artifact_path": str(artifact_path),
    "feature_count": len(bundle["feature_columns"]),
    "numeric_feature_count": len(bundle["numeric_columns"]),
    "random_seed": RANDOM_SEED,
    "threshold": 0.5,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model bundle to: {artifact_path}")
print(f"Saved metadata to: {metadata_path}")

Saved model bundle to: C:\Users\srina\Coding\Chess-Bot\lichess-bot\engines\train_eval\model_artifacts\policy_xgb.joblib
Saved metadata to: C:\Users\srina\Coding\Chess-Bot\lichess-bot\engines\train_eval\model_artifacts\policy_xgb_metadata.json


## Train Model

In [14]:
# Run evaluation and show feature importances after model training.
evaluate_split("Train", X_train_scaled, y_train)
evaluate_split("Validation", X_val_scaled, y_val)
evaluate_split("Test", X_test_scaled, y_test)

feature_importance = pd.Series(model.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=False)
print("\nTop 20 feature importances:")
print(feature_importance.head(20).to_string())



Train results
  Accuracy: 79.40%
  Balanced accuracy: 75.68%
  ROC-AUC: 0.8428
  Positive rate: 9.51%
  Confusion matrix:
[[1979900  486472]
 [  74935  184285]]
  Classification report:
              precision    recall  f1-score   support

           0     0.9635    0.8028    0.8758   2466372
           1     0.2747    0.7109    0.3963    259220

    accuracy                         0.7940   2725592
   macro avg     0.6191    0.7568    0.6361   2725592
weighted avg     0.8980    0.7940    0.8302   2725592


Validation results
  Accuracy: 79.42%
  Balanced accuracy: 75.28%
  ROC-AUC: 0.8380
  Positive rate: 9.51%
  Confusion matrix:
[[424874 103635]
 [ 16566  38981]]
  Classification report:
              precision    recall  f1-score   support

           0     0.9625    0.8039    0.8761    528509
           1     0.2733    0.7018    0.3934     55547

    accuracy                         0.7942    584056
   macro avg     0.6179    0.7528    0.6347    584056
weighted avg     0.8969   

## Split and Scale